In [1]:
from lib.backtest.FisslerZiegelBacktest import FisslerZiegelBacktest
from lib.dicts.index import Index
from lib.data.data_import_export import DataImporter, DataExporter
from lib.data.data_parser import ProcessData
from lib.forecast.roll_forecast import *
from lib.backtest.backtest import BacktestManager
from lib.auxiliares.metrics import calculate_metrics

I0000 00:00:1781513346.786563   18122 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1781513348.327563   18122 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [2]:
indexes = ['SAN.MC', 'BBVA.MC']# , 'SAB.MC', '^IBEX', 'BBVAE.MC', 'XTC5.MI', 'EURUSD=X']
input_method = 'yf'
start_get_data = '2021-10-04'
end_get_data = '2024-10-04'
start_calculation_date = '2024-09-04'
end_calculation_date = '2024-10-04'
confidence_level = 0.99
horizons = [1, 10]

In [3]:
index_instance = Index(indexes)
data_instance = DataImporter(indexes, start_get_data, end_get_data, input_method)
process_data_instance = ProcessData(index_instance, data_instance)

In [4]:
process_data_instance.fill_index_dict(
    confidence_level,
    start_calculation_date,
    end_calculation_date
)

In [5]:
data_dict = index_instance.get_index_dict()
forecast_dict = index_instance.get_forecast_dict()

In [6]:
forecast_instance = Forecast(data_dict, forecast_dict, start_calculation_date, end_calculation_date, horizons)
forecast_instance.run_forecast()


E0000 00:00:1781513386.576829   18534 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Progreso global forecasting: 100.00%. Executed for 3/3 volatilities of 2/2 indices 

In [8]:
calculate_metrics(forecast_dict, data_dict)

Metrics have been saved to: output/forecast_metrics.xlsx


In [9]:
backtest_manager = BacktestManager(data_dict, forecast_dict, confidence_level)
backtest_manager.run_backtest_multiquantile()
backtest_manager.run_backtest_rige()
backtest_manager.run_backtest_fisslerziegel()
backtest_dict = backtest_manager.get_backtest_dict()

In [10]:
exporter = DataExporter(data_dict, forecast_dict, backtest_dict)
exporter.export_to_excel()

El archivo se ha guardado en: output/backtest_results.xlsx


In [11]:
forecast_instance.clean_up_models()